In [1]:
def Segmentation (df,p,y):
    train_size = int(len(df) * p)
    train = df.iloc[:train_size]
    test = df.iloc[train_size:]
    X_train, y_train = train.drop(columns=[y]), train[y]
    X_test, y_test = test.drop(columns=[y]), test[y]
    return X_train, y_train, X_test, y_test

def RandomForest (X_train, y_train, X_test, train, test):
    modelo_rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
    modelo_rf.fit(X_train, y_train)
    train_pred = modelo_rf.predict(X_train) # No es necesario
    test_pred = modelo_rf.predict(X_test)
    indice_predicciones_rf = pd.Series(train.index.tolist() + test.index.tolist()) # revisar si va a ser util o como guardarlo de otra manera
    valores_predichos_rf = np.concatenate([train_pred, test_pred])
    return  indice_predicciones_rf, valores_predichos_rf

#def VisualizationModelResults(indice_predicciones_rf,valores_predichos_rf,train_size,y_train,y_test):
#    plt.figure(figsize=(14, 5))
#    plt.plot(indice_predicciones_rf, valores_predichos_rf, label='Predicción Árbol (train + test directo)', color='crimson')
#    plt.plot(df.index[:train_size], y_train, label='Train', color='royalblue')
#    plt.plot(df.index[train_size:], y_test, label='Test', color='darkorange')
#    plt.axvline(df.index[train_size], color='black', linestyle='--', label='Corte Train/Test')
#    plt.title("Predicción Random Forest: Train + Test con valores reales", fontsize=14)
#    plt.xlabel("Fecha")
#    plt.ylabel("Valor")
#    plt.legend()
#    plt.tight_layout()
#    plt.show()
#  NOS HACE FALTA??

def MetricsModels(y_train, y_test, train_pred, test_pred):
    metrics = {
        "Metric": ["MSE", "MAE Train", "RMSE Train", "R2 Train", "MAE Test", "RMSE Test", "R2 Test"],
        "Value": [
            mean_squared_error(y_test, test_pred),
            mean_absolute_error(y_train, train_pred),
            mean_squared_error(y_train, train_pred, squared=False),
            r2_score(y_train, train_pred),
            mean_absolute_error(y_test, test_pred),
            mean_squared_error(y_test, test_pred, squared=False),
            r2_score(y_test, test_pred)  # Corregí el error donde `rmse_test` se sobrescribía con `r2_score`
        ]
    }
    
    df_metrics = pd.DataFrame(metrics)  # Convertir los datos en un DataFrame
    return df_metrics




In [ ]:
def SarimaX(y_train,X_train,y_test,X_test,p,d,q,P,D,Q,s): ## hay que crear una funcion de parametros y llamarla dentro de la funcion
    # Entrenar modelo SARIMAX
    modelo_sarimax = SARIMAX(y_train, exog=X_train, order=(p, d, q), seasonal_order=(P, D, Q, s))
    resultado_sarimax = modelo_sarimax.fit()

    # Predicción con variables externas
    pred_sarimax_test = resultado_sarimax.forecast(steps=len(y_test), exog=X_test)
    pred_sarimax_train= resultado_sarimax.forecast(steps=len(y_train), exog=X_train) # No haría falta en el modelo
    return pred_sarimax_test

def Prophet(y_train,X_train,y_test,X_test):
    # Crear la columna 'ds' con las fechas del índice
    X_train["ds"] = X_train.index ## comprobar como vienen las fechas si en indice o en columna
    X_test["ds"]= X_test.index

    # De momento tenemos estos regresores, debemos definir los regresores cuando se decidan las variables exogenas
    #regresores_clima = [col for col in df.columns if col.startswith('conditions_')]
    #regresores_producto = [col for col in df.columns if col.startswith('CodigoArticulo_')]
    #todos_regresores = regresores_clima + regresores_producto

    #modelo = Prophet(
    #growth="linear",
    #changepoint_prior_scale=0.1,
    #seasonality_mode="multiplicative",
    #holidays_prior_scale=5.0,
    #seasonality_prior_scale=5.0,
    #n_changepoints=30
    #)

    modelo = Prophet()
    for r in regresores:
        modelo.add_regressor(r)

    modelo.fit(X_train)
    forecast_train = modelo.predict(X_train[['ds'] + regresores]) # No hace falta
    forecast_test = modelo.predict(X_test[['ds'] + regresores])
    return forecast_test



def ARIMAX(y_train,X_train,y_test,X_test,p,d,q):
    # Entrenar modelo ARIMAX (con variables externas X_train)
    #p, d, q = 1, 1, 1  # Ajustar según ACF/PACF
    modelo_arimax = ARIMA(y_train, exog=X_train, order=(p, d, q))
    resultado_arimax = modelo_arimax.fit()
    # Predicción
    pred_arimax_train = resultado_arimax.forecast(steps=len(y_train), exog=X_train) #No hace falta 
    pred_arimax_test = resultado_arimax.forecast(steps=len(y_test), exog=X_test)
    return pred_arimax_test







In [ ]:
import optuna


# trial es una instancia de optuna que sugiere los valores de los parametros en cada iteracion
# otra opcion seria incorporar manualmente a mano un rango
# suggest_in selecciona un entero de ese rango para analizar
# modelo para evaluar el modelo
# scores calculamos las puntuaciones
# seleccionamos el mejor


def objective(trial, X_train, y_train):
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    modelo = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split)
    
    # Validación cruzada
    scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="neg_mean_squared_error") #cv lo debemos escoger?
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20) #n_trials lo debemos escoger?


ModuleNotFoundError: No module named 'optuna'

In [ ]:
## Falta hacer un bucle para meter a todos los modelos en la validacion cruzada y escoger al mejor
